# Harfi: Llama 3.2 1B v5, part 2 of 2 (Colab)

v4's recipe (**Llama 3.2 1B**, LoRA rank 32, alpha 64, lr 2e-4) with 2 epochs instead of 3 (v4's best was epoch 2), on
the `harfi-v6` export (144,986 examples, spelling v2.0 + round 6), the data Qwen v2 trained on. About 4.8 hours on a T4,
so it trains in two parts: **part 1 on Kaggle** (`finetune_llama_v5.ipynb`, stops itself after about 2.5 hours with a
checkpoint) and **part 2 on Colab** (`finetune_llama_v5_colab.ipynb`, goes on from that checkpoint and makes the GGUF
files). A run stopped and resumed ends with the same weights as one uninterrupted run (checked on the laptop with a
tiny model).

**Before you start**, in Google Drive, folder `harfi` (My Drive -> harfi):
- `arabizi-train-v6.zip`, the training data (the same zip as for Kaggle);
- `llama-v5-part1.zip`, from part 1's Output on Kaggle.

**How to run:**
1. Runtime -> Change runtime type -> **T4 GPU**.
2. Runtime -> **Run all**, and allow access to Google Drive when asked. Keep the tab open.
3. If Colab disconnects or stops: reconnect and **Run all** again. Training goes on from the last checkpoint in Drive
   (saved every 538 steps, about 17 min), never from the start.
4. At the end, `harfi/out/<run>/` in Drive holds `arabizi-llama3.2-1b-v5-Q4_K_M.gguf`, `-Q8_0.gguf`, their `.sha256`
   files and `run.json`. Download those to be scored.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive

drive.mount("/content/drive")
HARFI = "/content/drive/MyDrive/harfi"


In [ ]:
REF = "data/round6-qwen-v2"  # the branch with checkpoints and resuming (main once it is merged)
# Fresh copy of the code on every run (an old clone would otherwise be kept and run).
!rm -rf /content/repo
!git clone --quiet --depth 1 --branch "$REF" https://github.com/Mustafa-Abbara/arabizi-keyboard /content/repo
!git -C /content/repo log -1 --format='repo commit: %h %s'
!pip install --quiet -r /content/repo/train/requirements.txt


In [ ]:
import glob
import json
import os
import subprocess
import sys
import zipfile

REPO = "/content/repo"
DATA = "/content/data"
if not os.path.exists(f"{DATA}/export.json"):
    zipfile.ZipFile(f"{HARFI}/arabizi-train-v6.zip").extractall(DATA)
with open(f"{DATA}/export.json") as f:
    summary = json.load(f)
print(summary["rows_total"], "examples | AI share", summary["ai_share"])
assert summary["rows_total"] == 144986, "this should be the harfi-v6 export (spelling v2.0 + round 6)"

# Checkpoints live in Drive, so a disconnect loses at most the steps since the last one.
OUT = f"{HARFI}/out"
BASE, TAG, NAME = "llama3.2-1b", "v5", "arabizi-llama3.2-1b-v5"
os.makedirs(OUT, exist_ok=True)
runs = sorted(glob.glob(f"{OUT}/*-{BASE}-lora-{TAG}"))
if not runs and os.path.exists(f"{HARFI}/llama-v5-part1.zip"):
    zipfile.ZipFile(f"{HARFI}/llama-v5-part1.zip").extractall(OUT)  # part 1 from Kaggle
    runs = sorted(glob.glob(f"{OUT}/*-{BASE}-lora-{TAG}"))
assert runs, "put llama-v5-part1.zip (part 1's Output on Kaggle) in Drive, folder harfi"
RUN = runs[-1]
print("going on from", RUN, sorted(os.listdir(f"{RUN}/checkpoints")) if os.path.isdir(f"{RUN}/checkpoints") else "")


In [ ]:
# Same settings as part 1. The merged model goes to the Colab disk, not Drive (2.5 GB, deleted after the export).
if os.path.exists(f"{RUN}/run.json"):
    print("training already finished")
else:
    print(f"=== {BASE} {TAG}, part 2", flush=True)
    subprocess.run([sys.executable, f"{REPO}/train/finetune_lora.py", "--base", BASE, "--data", DATA, "--out", OUT,
                    "--tag", TAG, "--rank", "32", "--alpha", "64", "--epochs", "2", "--save-steps", "538",
                    "--resume", RUN, "--merged-dir", "/content/merged"], check=True)
    assert not os.path.exists(f"{RUN}/PARTIAL.json"), "training is not finished: Run all again"
if not glob.glob(f"{RUN}/{NAME}-Q8_0.gguf"):
    assert os.path.isdir("/content/merged"), "the merged model is gone (new session): delete run.json in Drive and Run all"
    subprocess.run(["bash", f"{REPO}/train/export_gguf.sh", "/content/merged", NAME, RUN, "Q4_K_M Q8_0"], check=True)
    subprocess.run(["rm", "-rf", "/content/merged", f"{RUN}/checkpoints"], check=True)


## Results

Download the GGUF files, their `.sha256` files and `run.json` from `harfi/out/<run>/` in Drive.


In [ ]:
with open(f"{RUN}/run.json") as f:
    r = json.load(f)
print(f"{r['run_id']}: best validation loss {r['best_eval_loss']:.4f} ({r['minutes']} min over both parts)")
print([h for h in r["history"] if "eval_loss" in h])
for s in r["public_samples"]:
    print(f"  {s['arabizi']}\n    -> {s['output']}\n    want {s['reference']}")
!ls -l "$RUN"
